In [ ]:
import os

os.environ["RAY_DEDUP_LOGS"] = "0"
os.environ["RAY_DISABLE_METRICS"] = "1"
os.environ["RAY_metrics_agent_port"] = "-1"

import ray

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
from autotsc import utils

In [ ]:
dataset = "CricketX"
dataset = "GestureMidAirD1"
# dataset = 'Symbols'
dataset = "ProximalPhalanxOutlineCorrect"
# dataset = 'DiatomSizeReduction' # !!!!!!!!!1check if this works
# dataset = 'Crop'

X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
from autotsc.models2 import StackerV4Ray

In [ ]:
s = StackerV4Ray(
    random_state=270, n_repetitions=1, k_folds=10, time_limit_in_seconds=None, n_jobs=-1
)
s.fit(X_train, y_train)

In [ ]:
pred = s.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"Accuracy on {dataset}: {acc}")

In [ ]:
# Accuracy on ProximalPhalanxOutlineCorrect: 0.9209621993127147
# Accuracy on ProximalPhalanxOutlineCorrect: 0.9209621993127147

In [ ]:
X = s.get_Xt()
X

In [ ]:
X.estimated_size('gb')

In [ ]:
import polars as pl
X.with_columns(pl.col("*").cast(pl.Float16)).estimated_size('gb')

In [ ]:
ray.shutdown()

In [ ]:
rocket = s.get_model("rocket")
rocket

In [ ]:
from aeon.transformations.collection.convolution_based import Rocket

In [ ]:
rocket = Rocket(n_jobs=-1)
rocket.fit(X_train, y_train)

In [ ]:
Xt = rocket.transform(X_train)
Xt.shape